# ⚡ Hardware Roofline Profiling & OpenAI Triton Kernel Acceleration

Welcome to **NeuralOps v1.3.7**'s Hardware Roofline & Kernel Optimization Guide!

In high-performance deep learning, raw algorithmic complexity $\mathcal{O}(T)$ vs. $\mathcal{O}(T^2)$ is only half the story. Real-world performance on modern GPU architectures (e.g., Ada Lovelace, Hopper, Blackwell) is fundamentally governed by the **Hardware Roofline Model**:

$$\text{Attainable Performance [TFLOPS]} = \min\left(\text{Peak Compute Ceiling}, \text{Arithmetic Intensity} \times \text{Peak Memory Bandwidth}\right)$$

where **Arithmetic Intensity** ($AI$) is defined as:
$$\text{Arithmetic Intensity} = \frac{\text{Total Floating-Point Operations (FLOPs)}}{\text{Total DRAM / HBM Data Movement (Bytes)}} \quad \left[\frac{\text{FLOPs}}{\text{Byte}}\right]$$

### The Memory Wall & SRAM Fusion Strategy
Standard attention implementations execute multiple memory round-trips to High Bandwidth Memory (HBM) for intermediate tensors ($Q K^\top$, softmax normalization, dropout, matrix multiplications). Because HBM bandwidth ($\sim 256 \text{ GB/s} - 3 \text{ TB/s}$) is orders of magnitude slower than on-chip SRAM register bandwidth ($\sim 15 - 30 \text{ TB/s}$), standard attention quickly stalls in the **Memory-Bound Regime**.

NeuralOps implements **26 compiled OpenAI Triton kernels** that fuse recurrent state updates directly in on-chip SRAM, bypassing HBM roundtrips and catapulting execution into the **Compute-Bound Regime**.

---
### What this Tutorial Covers:
1. **Physical GPU Architecture Profiling**: Measuring empirical SM architecture, peak TFLOPS, HBM bandwidth, and hardware ridge point $\left(I^* = \frac{\text{Peak FLOPs}}{\text{Peak Bandwidth}}\right)$.
2. **Empirical Roofline Analysis**: Profiling `gated_delta_net` and `gla` with `neuralops.profile_roofline`.
3. **Ascii & High-DPI Visualizations**: Rendering ASCII roofline diagrams with `render_roofline_terminal` and plotting multi-kernel empirical roofline curves in Matplotlib.
4. **Benchmarking Compiled Triton Kernels**: Validating speedups and numerical parity ($\Delta < 10^{-2}$) across Triton scan implementations vs. un-fused PyTorch fallbacks.
5. **Ledger Auditing**: Verifying logged performance records in `neuralops.get_default_ledger()`.

In [ ]:
import torch
import neuralops
from neuralops.core.roofline import profile_roofline, render_roofline_terminal
import matplotlib.pyplot as plt
import numpy as np

print(f"NeuralOps Version: {neuralops.__version__}")
print(f"PyTorch Version:   {torch.__version__}")
print(f"CUDA Available:    {torch.cuda.is_available()}")

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"Active GPU:        {props.name}")
    print(f"Compute Cap:       SM {props.major}.{props.minor}")
    print(f"Total VRAM:        {props.total_memory / (1024**3):.2f} GB")
    print(f"SM MultiProc Count:{props.multi_processor_count}")

## 1. Physical Hardware Roofline Profiling
We use `neuralops.profile_roofline` to profile `gated_delta_net` under a representative sequence configuration ($B=2, T=1024, D=768, H=12$).

The profiler accurately measures:
- **Total Algorithmic Work**: Theoretical FLOPs for the layer
- **Naive vs. Fused HBM Traffic**: Total DRAM read/write bytes required by standard PyTorch vs. SRAM-fused Triton
- **Arithmetic Intensity Shift**: $\Delta AI = \frac{\text{FLOPs}}{\text{Bytes}_{\text{fused}}} - \frac{\text{FLOPs}}{\text{Bytes}_{\text{naive}}}$
- **Empirical Hardware Speedup**: $\frac{t_{\text{pytorch}}}{t_{\text{triton}}}$

In [ ]:
device_target = "cuda:0" if torch.cuda.is_available() else "cpu"

report_gdn = profile_roofline(
    module_name="gated_delta_net",
    batch_size=2,
    seq_len=1024,
    embed_dim=768,
    num_heads=12,
    device=device_target,
    warmup=5,
    rep=10,
    log_to_ledger=True,
)

print(f"Hardware Roofline Profiling Results for 'gated_delta_net':")
print(f"  • Module / Precision:      {report_gdn.module_name} ({report_gdn.precision})")
print(f"  • Target GPU:              {report_gdn.device_name} ({report_gdn.sm_version})")
print(f"  • Total Algorithmic Work:  {report_gdn.total_flops / 1e9:.2f} GFLOPs")
print(f"  • Naive HBM Traffic:       {report_gdn.hbm_bytes_naive / (1024**2):.2f} MB  (AI: {report_gdn.arithmetic_intensity_naive:.1f} FLOPs/B)")
print(f"  • Fused HBM Traffic:       {report_gdn.hbm_bytes_fused / (1024**2):.2f} MB  (AI: {report_gdn.arithmetic_intensity_fused:.1f} FLOPs/B)")
print(f"  • On-Chip SRAM Saved:      {report_gdn.sram_bytes_saved_mb:.2f} MB")
print(f"  • PyTorch Native Latency:  {report_gdn.latency_pytorch_ms:.3f} ms")
print(f"  • NeuralOps Triton Latency:{report_gdn.latency_triton_ms:.3f} ms")
print(f"  • Measured Speedup:        {report_gdn.speedup_x:.2f}x")
print(f"  • Identified Bottleneck:   {report_gdn.bottleneck_regime}")

## 2. ASCII Roofline Visualization
NeuralOps formats physical roofline metrics into an ASCII chart depicting the hardware characteristic boundary, slope (bandwidth), and empirical operating points ($P$ = PyTorch fallback, $T$ = Fused Triton kernel).

In [ ]:
chart_ascii = render_roofline_terminal(report_gdn)
print(chart_ascii)

## 3. High-DPI Publication-Grade Roofline Plot
To compare multiple attention kernels against the theoretical hardware ceiling, we profile Gated Linear Attention (`gla`) alongside `gated_delta_net` and render their positions on the continuous Log-Log Roofline characteristic curve.

In [ ]:
# Profile additional kernel: Gated Linear Attention (GLA)
report_gla = profile_roofline(
    module_name="gla",
    batch_size=2,
    seq_len=1024,
    embed_dim=768,
    num_heads=12,
    device=device_target,
    warmup=5,
    rep=10,
    log_to_ledger=True,
)

# Plot Theoretical Roofline Curve and Operating Points
peak_tflops = report_gdn.peak_tflops
peak_bw = report_gdn.peak_bandwidth_gb_s
ridge = report_gdn.ridge_point

# Intensity domain [0.1, 2048]
ai_grid = np.logspace(-1, 3.5, 400)
roofline_tflops = np.minimum(peak_tflops, (peak_bw * ai_grid) / 1e3)

plt.figure(figsize=(10, 5.5), dpi=130)
plt.plot(ai_grid, roofline_tflops, color="#2c3e50", linewidth=2.5, label=f"Hardware Roofline Ceiling ({report_gdn.device_name})")
plt.axvline(x=ridge, color="#e74c3c", linestyle=":", label=f"Hardware Ridge Point ({ridge:.1f} FLOPs/B)")

# Plot operating points
plt.scatter([report_gdn.arithmetic_intensity_naive], [report_gdn.achieved_tflops / report_gdn.speedup_x], 
            color="#e74c3c", s=90, marker="x", label=f"GDN PyTorch Native ({report_gdn.latency_pytorch_ms:.1f} ms)")
plt.scatter([report_gdn.arithmetic_intensity_fused], [report_gdn.achieved_tflops], 
            color="#27ae60", s=110, marker="o", label=f"GDN NeuralOps Triton ({report_gdn.latency_triton_ms:.1f} ms, {report_gdn.speedup_x:.1f}x)")

plt.scatter([report_gla.arithmetic_intensity_naive], [report_gla.achieved_tflops / max(1.0, report_gla.speedup_x)], 
            color="#f39c12", s=90, marker="^", label=f"GLA PyTorch Native ({report_gla.latency_pytorch_ms:.1f} ms)")
plt.scatter([report_gla.arithmetic_intensity_fused], [report_gla.achieved_tflops], 
            color="#2980b9", s=110, marker="s", label=f"GLA NeuralOps Triton ({report_gla.latency_triton_ms:.1f} ms, {report_gla.speedup_x:.1f}x)")

plt.xscale("log")
plt.yscale("log")
plt.xlabel("Arithmetic Intensity (FLOPs / Byte)", fontsize=11, fontweight="bold")
plt.ylabel("Attainable Performance (TFLOPS)", fontsize=11, fontweight="bold")
plt.title(f"Empirical Hardware Roofline Model: PyTorch Native vs. Triton Fused Kernels\n({report_gdn.device_name} — {report_gdn.sm_version})", fontsize=12, fontweight="bold")
plt.legend(frameon=True, fontsize=9.5, loc="lower right")
plt.grid(True, which="both", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

## 4. Deep-Dive: Triton Kernel Execution & Numerical Parity
NeuralOps provides 26 optimized Triton kernels. A core operational invariant is:
1. **Numerical Parity**: Max difference $\Delta < 0.01$ (and Cosine Similarity $\approx 1.0$) between the Triton compiled kernel and its PyTorch reference.
2. **Speedup Guarantee**: $\ge 1.0\times$ speedup over un-fused PyTorch fallback.

Let's test `gla_scan_triton` directly against PyTorch recurrent formulation with synthetic input states.